# RAG Chatbot with Groq — Multi-PDF Q&A with Conversation History

Simple pipeline: **3 PDFs → chunks (tagged by source) → embeddings → one shared FAISS index → Groq LLM**,
wrapped in a chat loop that remembers prior turns and can tell the documents apart.

Documents used here: Mahesh Babu, Kamal Haasan, and Amitabh Bachchan biographies — but any PDFs work.

Run cells top to bottom.

## 1. Install dependencies

In [ ]:
!pip install -q groq sentence-transformers faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 9.0 MB/s eta 0:00:00


## 2. Set your Groq API key
Get a free key at https://console.groq.com

In [ ]:
import os
from getpass import getpass

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


## 3. Upload the PDFs

In [ ]:
from google.colab import files

uploaded = files.upload()  # select all 3: Mahesh Babu, Kamal Haasan, Amitabh Bachchan PDFs
pdf_paths = list(uploaded.keys())
print(f"Loaded {len(pdf_paths)} PDFs: {pdf_paths}")

Saving Krishna_and_Mahesh_Babu_Legacy (1).pdf to Krishna_and_Mahesh_Babu_Legacy (1).pdf
Loaded 1 PDFs: ['Krishna_and_Mahesh_Babu_Legacy (1).pdf']


## 4. Extract text and chunk it — now tagged by source document

Each chunk now carries a `source` field (the filename it came from) alongside `page`. This is what
lets the system tell the three documents apart later, even though retrieval searches across all of
them together rather than picking a document first.

In [ ]:
import re
from pypdf import PdfReader

def load_and_chunk(pdf_path, chunk_size=800, overlap=150):
    reader = PdfReader(pdf_path)
    chunks = []
    for page_num, page in enumerate(reader.pages, start=1):
        text = re.sub(r"\s+", " ", (page.extract_text() or "")).strip()
        start = 0
        while start < len(text):
            piece = text[start:start + chunk_size]
            if piece:
                chunks.append({"text": piece, "page": page_num, "source": pdf_path})
            start += chunk_size - overlap
    return chunks

chunks = []
for path in pdf_paths:
    doc_chunks = load_and_chunk(path)
    chunks.extend(doc_chunks)
    print(f"{path}: {len(doc_chunks)} chunks")

for i, c in enumerate(chunks):
    c["id"] = i

print(f"\nTotal: {len(chunks)} chunks across {len(pdf_paths)} documents")

Krishna_and_Mahesh_Babu_Legacy (2).pdf: 134 chunks
Kamal_Haasan_Legacy.pdf: 98 chunks
Amitabh_Bachchan_Legacy.pdf: 84 chunks

Total: 316 chunks across 3 documents


## 5. Embed chunks and build a FAISS index

`all-MiniLM-L6-v2` is a small, fast, free local embedding model — no API call needed for this step.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)
faiss.normalize_L2(embeddings)  # so inner product == cosine similarity

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

print(f"Indexed {index.ntotal} vectors of dimension {embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Indexed 316 vectors of dimension 384


## 6. Retrieval function

Note there's no "pick a document first" step — the query is embedded once and compared against
every chunk from all 3 PDFs in the same index. The right document's chunks simply score higher
for a question about that actor, because that's where the semantically closest text lives.

In [ ]:
def retrieve(query, top_k=4):
    q_vec = embed_model.encode([query], convert_to_numpy=True)
    faiss.normalize_L2(q_vec)
    scores, idxs = index.search(q_vec, top_k)
    return [
        {**chunks[i], "score": float(s)}
        for s, i in zip(scores[0], idxs[0]) if i != -1
    ]

## 7. Groq-powered answer generation, with conversation history

Each retrieved chunk still carries its `source` filename and `page` — that's what lets the model
tell "which document is this from" when a question needs it, even though we're not forcing a
citation into every answer.

`history` is a running list of `{"role": ..., "content": ...}` messages, carried across turns so the
model can resolve follow-ups like "what about the other actor?" using earlier context.

In [ ]:
from groq import Groq

client = Groq(api_key=os.environ["GROQ_API_KEY"])

SYSTEM_PROMPT = (
    "You are a helpful assistant answering questions about a set of PDF documents, each about a "
    "different actor. Use ONLY the provided context to answer. If a question is about a specific "
    "actor, use only the context chunks from that actor's document. If the answer isn't in the "
    "context, say 'I don't have enough information in these documents to answer that.' Keep answers "
    "concise."
)

history = []  # conversation memory, persists across turns in this session

def chat(query, top_k=5):
    retrieved = retrieve(query, top_k=top_k)
    context_block = "\n\n".join(
        f"[source: {c['source']}, page {c['page']}] {c['text']}" for c in retrieved
    )

    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(history)
    messages.append({
        "role": "user",
        "content": f"CONTEXT:\n{context_block}\n\nQUESTION: {query}"
    })

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        temperature=0.1,
    )
    answer = response.choices[0].message.content

    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": answer})

    return answer

## 8. Try it — including a cross-document question

The first two questions target one document each; the third forces the model to pull from all
three and keep them straight.

In [ ]:
print(chat("What awards has Mahesh Babu won?"))

Mahesh Babu has won the following awards: 
- 9 Nandi Awards 
- 5 Filmfare Awards South for Best Actor – Telugu 
- 4 SIIMA Awards 
- 3 CineMAA Awards 
- 3 Santosham Film Awards 
- 2 Gaddar Telangana Film Awards 
- 1 IIFA Utsavam Award


In [ ]:
print(chat("What about Kamal Haasan — how many National Film Awards does he have?"))

Kamal Haasan has won 5 National Film Awards.


In [ ]:
print(chat("Compare how Amitabh Bachchan, Kamal Haasan, and Mahesh Babu each recovered from a career setback."))

I don't have enough information in these documents to answer that. The documents only mention Amitabh Bachchan's recovery from the ABCL collapse through Kaun Banega Crorepati, but do not provide information on career setbacks or recoveries for Kamal Haasan or Mahesh Babu.


In [ ]:
print(chat("Which of the three actors has been in politics, and what happened?"))

Amitabh Bachchan and Kamal Haasan have been involved in politics. Amitabh Bachchan had a period in politics, but the details of his political career are not specified in the provided context. Kamal Haasan entered electoral politics in 2018, with his reform-oriented messaging being compared to MGR's more populist approach, but the outcome of his political endeavors is not mentioned in the context. There is no information about Mahesh Babu being involved in politics.


## 9. Interactive chat loop (optional)

Run this cell and type questions directly. Type `exit` to stop.

In [ ]:
while True:
    q = input("\nYou: ")
    if q.strip().lower() in ("exit", "quit"):
        break
    print("Bot:", chat(q))


You: exit


## 10. Reset conversation history (optional)

Run this if you want to start a fresh conversation without re-uploading or re-indexing the PDF.

In [ ]:
history.clear()
print("Conversation history cleared.")

NameError: name 'history' is not defined